In [ ]:
import pandas as pd
import numpy as np

--- Cek Nilai Kosong Standar ---
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

--- Deteksi Teks Anomali ---
Kolom 'Item': terdapat 636 baris ERROR/UNKNOWN
Kolom 'Quantity': terdapat 341 baris ERROR/UNKNOWN
Kolom 'Price Per Unit': terdapat 354 baris ERROR/UNKNOWN
Kolom 'Total Spent': terdapat 329 baris ERROR/UNKNOWN
Kolom 'Payment Method': terdapat 599 baris ERROR/UNKNOWN
Kolom 'Location': terdapat 696 baris ERROR/UNKNOWN
Kolom 'Transaction Date': terdapat 301 baris ERROR/UNKNOWN


In [4]:
# Muat data ke dalam Pandas
df_cafe = pd.read_csv('dirty_cafe_sales.csv')

# Periksa jumlah baris kosong (Null/NaN) standar
print("--- Cek Nilai Kosong Standar ---")
print(df_cafe.isna().sum())

# Deteksi anomali teks tersembunyi ("ERROR" atau "UNKNOWN")
print("\n--- Deteksi Teks Anomali ---")
for col in df_cafe.columns:
    anomaly_count = df_cafe[col].astype(str).str.contains('ERROR|UNKNOWN', na=False).sum()
    if anomaly_count > 0:
        print(f"Kolom '{col}': terdapat {anomaly_count} baris ERROR/UNKNOWN")

--- Cek Nilai Kosong Standar ---
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

--- Deteksi Teks Anomali ---
Kolom 'Item': terdapat 636 baris ERROR/UNKNOWN
Kolom 'Quantity': terdapat 341 baris ERROR/UNKNOWN
Kolom 'Price Per Unit': terdapat 354 baris ERROR/UNKNOWN
Kolom 'Total Spent': terdapat 329 baris ERROR/UNKNOWN
Kolom 'Payment Method': terdapat 599 baris ERROR/UNKNOWN
Kolom 'Location': terdapat 696 baris ERROR/UNKNOWN
Kolom 'Transaction Date': terdapat 301 baris ERROR/UNKNOWN


In [11]:
df_cafe['Quantity'] = pd.to_numeric(df_cafe['Quantity'], errors='coerce')
df_cafe['Price Per Unit'] = pd.to_numeric(df_cafe['Price Per Unit'], errors='coerce')
df_cafe['Total Spent'] = pd.to_numeric(df_cafe['Total Spent'], errors='coerce')

print(df_cafe[['Quantity', 'Price Per Unit', 'Total Spent']].head(5))

   Quantity  Price Per Unit  Total Spent
0       2.0             2.0          4.0
1       4.0             3.0         12.0
2       4.0             1.0          NaN
3       2.0             5.0         10.0
4       2.0             2.0          4.0


In [13]:
df_cafe['Quantity'] = df_cafe['Quantity'].fillna(df_cafe['Total Spent'] / df_cafe['Price Per Unit'])

df_cafe['Price Per Unit'] = df_cafe['Price Per Unit'].fillna(df_cafe['Total Spent'] / df_cafe['Quantity'])

df_cafe['Total Spent'] = df_cafe['Total Spent'].fillna(df_cafe['Quantity'] * df_cafe['Price Per Unit'])

df_cafe=df_cafe.dropna(subset=['Quantity','Price Per Unit', 'Total Spent'])

print(df_cafe[['Quantity', 'Price Per Unit', 'Total Spent']].isna().sum())

Quantity          0
Price Per Unit    0
Total Spent       0
dtype: int64


In [14]:
kolom_teks = ['Item', 'Location', 'Payment Method']

for col in kolom_teks:
    df_cafe[col] = df_cafe[col].replace(['UNKNOWN', 'ERROR'], 'Other')
    df_cafe[col] = df_cafe[col].fillna('Other')

df_cafe['Transaction Date'] = pd.to_datetime(df_cafe['Transaction Date'], errors='coerce')

df_cafe = df_cafe.dropna(subset=['Transaction Date'])

print("Distribusi Metode Pembayaran saat ini:")
print(df_cafe['Payment Method'].value_counts())
print(f"\nTotal baris data yang tersisa dan bersih: {len(df_cafe)}")

Distribusi Metode Pembayaran saat ini:
Payment Method
Other             2997
Digital Wallet    2186
Credit Card       2157
Cash              2145
Name: count, dtype: int64

Total baris data yang tersisa dan bersih: 9485


C:\Users\torik\AppData\Local\Temp\ipykernel_9112\748796517.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cafe[col] = df_cafe[col].replace(['UNKNOWN', 'ERROR'], 'Other')
C:\Users\torik\AppData\Local\Temp\ipykernel_9112\748796517.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cafe[col] = df_cafe[col].fillna('Other')
C:\Users\torik\AppData\Local\Temp\ipykernel_9112\748796517.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_ind

In [18]:
jumlah_duplikat = df_cafe.duplicated(subset=['Transaction ID']).sum()
print(f"Ditemukan {jumlah_duplikat} baris transaksi duplikat.")

df_cafe = df_cafe.drop_duplicates(subset=['Transaction ID'], keep='first')

nama_file_baru = 'cleaned_cafe_sales.csv'
df_cafe.to_csv(nama_file_baru, index=False)

print(f"\nPembersihan selesai! Data berhasil disimpan sebagai '{nama_file_baru}'.")
print(f"Total baris data akhir yang siap dianalisis: {len(df_cafe)}")

Ditemukan 0 baris transaksi duplikat.

Pembersihan selesai! Data berhasil disimpan sebagai 'cleaned_cafe_sales.csv'.
Total baris data akhir yang siap dianalisis: 9485


In [32]:
total_revenue = df_cafe['Total Spent'].sum()
print(f"Total Pendapatan Kafe: ${total_revenue:,.2f}\n")

top_items = df_cafe.groupby('Item')['Total Spent'].sum().sort_values(ascending=False)
print("--- Pendapatan per Menu ---")
print(top_items, "\n")

location_sales = df_cafe.groupby('Location')['Total Spent'].sum().sort_values(ascending=False)
print("--- Pendapatan berdasarkan Lokasi ---")
print(location_sales, "\n")

location_sales = df_cafe.groupby('Item')['Quantity'].sum().sort_values(ascending=False)
print("--- Most sold item ---")
print(location_sales, "\n")

df_cafe['Month'] = df_cafe['Transaction Date'].dt.to_period('M')
monthly_sales = df_cafe.groupby('Month')['Total Spent'].sum()
print("--- Pendapatan per Bulan ---")
print(monthly_sales)

Total Pendapatan Kafe: $84,643.00

--- Pendapatan per Menu ---
Item
Salad       16545.0
Sandwich    12996.0
Smoothie    12736.0
Juice       10017.0
Cake         9807.0
Other        8037.5
Coffee       6800.0
Tea          4639.5
Cookie       3065.0
Name: Total Spent, dtype: float64 

--- Pendapatan berdasarkan Lokasi ---
Location
Other       33571.0
In-store    25784.5
Takeaway    25287.5
Name: Total Spent, dtype: float64 

--- Most sold item ---
Item
Coffee      3400.0
Juice       3339.0
Salad       3309.0
Cake        3269.0
Sandwich    3249.0
Smoothie    3184.0
Tea         3093.0
Cookie      3065.0
Other       2756.0
Name: Quantity, dtype: float64 

--- Pendapatan per Bulan ---
Month
2023-01    7219.0
2023-02    6626.0
2023-03    7190.5
2023-04    7168.0
2023-05    6916.5
2023-06    7350.0
2023-07    6876.5
2023-08    7049.5
2023-09    6842.0
2023-10    7295.0
2023-11    6957.0
2023-12    7153.0
Freq: M, Name: Total Spent, dtype: float64
